# cAPTure: Gate-0 data audit

Run this notebook in Google Colab with a **CPU runtime**. It audits development scenarios before selecting features, graph endpoints, or a window duration. It does not train models or access final-test scenarios.

The workflow uses the authors' public Drive for source CSVs and your mounted Drive for durable results, and local Colab disk for conversion and SQL aggregation. Process one scenario at a time. Start with `SMOKE`; `FULL_DEV` requires a reviewed smoke run.

The audit Parquet preserves every source row and raw column alongside diagnostic metadata. It is **not** a frozen, model-ready canonical dataset. No features are selected, labels guessed, or duplicate packets removed.


## 1. Mount Drive and load the project

Before running, push the implementation to the configured Git branch, or place an updated repository copy in Drive and set `PROJECT_ROOT` to that path. This notebook imports the repository module; it is not self-contained.

The manifest contains verified development CSV IDs, filenames, and byte sizes from the authors' [pre-merged training scenario folder](https://drive.google.com/drive/folders/1f8koJbuPs0_CWoItWdt3VF-lZ_R6JAms). No raw CSV copies are needed in your Drive. Colab downloads individual files by ID; it never recursively downloads the folder.

Folder metadata verification does not prove packet-level integrity. That is the purpose of Gate 0.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_ROOT = Path("/content/capture_gate0_work")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)], check=True
    )

required_files = [
    PROJECT_ROOT / "code/python/utils/capture_data.py",
    PROJECT_ROOT / "code/python/requirements-capture.txt",
    PROJECT_ROOT / "configs/capture_experiment_v1.yaml",
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Update the repository copy first: {missing_files}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture.txt")], check=True
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
print("CPU audit environment is ready.")


Mounted at /content/drive
CPU audit environment is ready.


## 2. Run the small synthetic checks

Run these checks before downloading or processing large files. They cover half-open windows, repeated packets, cross-chunk timestamp inversions, unknown labels, held-out exclusion, and smoke-review integrity. They use temporary synthetic data and do not inspect cAPTure contents.

These checks have not been executed in the development workspace; this Colab run is the first runtime validation.


In [2]:
test_environment = dict(os.environ)
test_environment["PYTHONPATH"] = str(PROJECT_ROOT / "code/python")
subprocess.run(
    [sys.executable, "-m", "unittest", "discover",
     "-s", str(PROJECT_ROOT / "code/python/tests"),
     "-p", "test_capture_data.py", "-v"],
    env=test_environment, cwd=PROJECT_ROOT, check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', 'discover', '-s', '/content/temporalgnn-nids/code/python/tests', '-p', 'test_capture_data.py', '-v'], returncode=0)

## 3. Configure the audit and official downloads

Start with `SMOKE` (`train_empty_conn` and `train_dollar_char`). The source configuration is built from the manifest. Only development author-train scenarios are permitted. Your Drive stores reports and compressed audit Parquet; raw CSVs are staged on local Colab disk.

For `FULL_DEV`, supply a smoke-review file from section 8. A unique run directory prevents overwriting earlier results. Leave the chunk size at 25,000 initially; reduce it if wide CSV columns exhaust RAM. DuckDB is limited to two threads and 2 GB, but pandas and Arrow also need memory.

Google Drive may temporarily impose download quotas. If a download fails, the error is preserved; no alternative dataset is substituted.


In [3]:
from datetime import datetime, timezone
import json
import shutil
import pandas as pd
from IPython.display import display
from utils.capture_data import (
    AuditSchema, inspect_csv, load_manifest, run_gate0, selected_scenarios, stage_source,
    sha256_file, validate_smoke_review, write_json,
)

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
MANIFEST = load_manifest(MANIFEST_PATH)
#MODE = "SMOKE"
MODE = "FULL_DEV"
SCENARIOS = selected_scenarios(MANIFEST, MODE)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + MODE.lower()
DRIVE_RUN_DIR = DRIVE_ROOT / "runs" / RUN_ID

CHUNK_SIZE = 25_000
DUCKDB_MEMORY_LIMIT = "2GB"
DUCKDB_THREADS = 2
KEEP_AUDIT_PARQUET = True
#SMOKE_REVIEW_PATH = None  # Set the reviewed smoke JSON path before FULL_DEV.
SMOKE_REVIEW_PATH = "/content/drive/MyDrive/capture_gate0/runs/20260917T155516_689141Z_smoke/smoke_review.json"


SOURCE_CACHE_ROOT = LOCAL_ROOT / RUN_ID / "sources"
SOURCES = {}
for scenario in SCENARIOS:
    entry = MANIFEST["scenarios"][scenario]
    required = ("source_file_id", "expected_filename", "expected_size_bytes")
    if any(entry.get(key) is None for key in required):
        raise ValueError(f"Source metadata is incomplete for {scenario}. Update the manifest.")
    SOURCES[scenario] = {
        **{key: entry[key] for key in required},
        "metadata_verified": True,
    }

if MODE == "FULL_DEV":
    if SMOKE_REVIEW_PATH is None:
        raise ValueError("Set SMOKE_REVIEW_PATH before inspecting FULL_DEV sources.")
    validate_smoke_review(Path(SMOKE_REVIEW_PATH), sha256_file(MANIFEST_PATH), MANIFEST)

print(f"Mode: {MODE}")
print(f"Scenarios: {SCENARIOS}")
print(f"Output directory: {DRIVE_RUN_DIR}")
print(f"Free local storage: {shutil.disk_usage(LOCAL_ROOT).free / 1024**3:.1f} GiB")


Mode: FULL_DEV
Scenarios: ['train_empty_conn', 'train_qos_mid', 'train_dollar_char', 'train_slash_char', 'train_sub_exf']
Output directory: /content/drive/MyDrive/capture_gate0/runs/20260917T201026_302809Z_full_dev
Free local storage: 87.3 GiB


## 4. Download and inspect the first scenario

Download the first selected CSV to local Colab disk, then read a prefix of 2,000 rows. The completed download is bound to its source configuration and a local SHA-256 receipt. The audit reuses it after verifying the receipt, so it is not downloaded twice.

Only one scenario is staged for preview. Remaining schemas are checked against their CSV headers when each scenario is processed. Missing attack labels in the preview do not mean the full scenario contains no attacks.

All raw columns are initially read as strings to preserve identifiers, leading zeros, empty fields, and mixed types. The preview is not used to fit preprocessing or select features.


In [ ]:
CSV_SEPARATOR = ","
CSV_ENCODING = "utf-8-sig"
for source in SOURCES.values():
    source["separator"] = CSV_SEPARATOR
    source["encoding"] = CSV_ENCODING

PREVIEW_SCENARIO = SCENARIOS[0]
preview_path = stage_source(
    SOURCES[PREVIEW_SCENARIO], SOURCE_CACHE_ROOT / PREVIEW_SCENARIO,
)
inspection = inspect_csv(
    preview_path, separator=CSV_SEPARATOR,
    encoding=CSV_ENCODING, sample_rows=2_000,
)
INSPECTIONS = {PREVIEW_SCENARIO: inspection}
print(f"{PREVIEW_SCENARIO}: {inspection['source_size_bytes'] / 1024**3:.2f} GiB")
display(pd.DataFrame([
    {"column": column, "prefix_examples": values}
    for column, values in inspection["sample_values"].items()
]))


Downloading...
From (original): https://drive.google.com/uc?id=1nfR1RRZO3jMG_A9KDY8wyr0A0rVlHiBN
From (redirected): https://drive.google.com/uc?id=1nfR1RRZO3jMG_A9KDY8wyr0A0rVlHiBN&confirm=t&uuid=fd7fd1ef-76a1-435d-a747-49107894472b
To: /content/capture_gate0_work/20260917T170658_227477Z_full_dev/sources/train_empty_conn/normal_empty_conn_train.csv.part
100%|██████████| 1.03G/1.03G [00:28<00:00, 36.7MB/s]


train_empty_conn: 0.96 GiB


,column,prefix_examples
0,layers_frame_frame.section_number,[1]
1,timestamp,"[1970-01-01 01:00:00.019792+01:00, 1970-01-01 ..."
2,layers_frame_frame.time_epoch,"[0.019792, 0.036245, 0.03809, 0.051729, 0.0682..."
3,layers_frame_frame.number,"[1, 2, 3, 4, 5, 6, 7, 8]"
4,layers_frame_frame.len,"[114, 90, 111, 64, 78, 70, 100, 74]"
...,...,...
103,phase_idx,[]
104,phase_name,[]
105,phase_number,[]
106,step_number,[]


## 5. Confirm the declared schema

The mapping below follows the authors' scenario-construction notebooks:

- `timestamp` is the post-merge timestamp. The relative frame epoch is not used because the merge adjusts `timestamp`.
- `label` contains `normal` or the attack command. It is also the most specific published attack-step name.
- `phase_name` and `sequence_id` retain the authors' evaluation annotations.
- Ethernet source and destination MAC fields are provisional Gate-0 endpoints because they cover IP, IPv6, ARP, and other Ethernet traffic. Gate 0 will determine whether this remains the graph node key.

The raw-to-binary mappings enumerate the values reported by the official merge notebooks for all five development author-train scenarios. Any additional value remains unmapped and blocks progression. Endpoint identifiers are topology metadata and are not approved as model features.

The diagnostic iteration key is `(scenario, attack_step, phase, sequence_id)`. Its scientific meaning must still be reviewed after the audit.


In [ ]:
COMMON_SCHEMA = {
    "timestamp": "timestamp",
    "timestamp_unit": "datetime",
    "label": "label",
    "source_endpoint": "layers_eth_eth.src",
    "destination_endpoint": "layers_eth_eth.dst",
    "attack_step": "label",
    "phase": "phase_name",
    "sequence_id": "sequence_id",
    "separator": CSV_SEPARATOR,
    "encoding": CSV_ENCODING,
}
SCHEMA_OVERRIDES = {
    "train_empty_conn": {
        "label_mapping": {
            "normal": 0,
            "nmap_10_T4": 1,
            "brute_force_timing": 1,
            "empty_conn_ddos": 1,
            "nmap_mqtt": 1,
            "nmap_banner": 1,
            "empty_conn": 1,
            "nmap_sub": 1,
            "mqtt_cat": 1,
            "sftp_inst": 1,
        },
    },
    "train_dollar_char": {
        "label_mapping": {
            "normal": 0,
            "dollar_char": 1,
            "nmap_10_T5": 1,
            "nmap_mqtt": 1,
            "brute_force_malformed": 1,
            "nmap_banner": 1,
            "nmap_sub": 1,
            "mqtt_cat": 1,
            "scp_inst": 1,
        },
    },
    "train_qos_mid": {
        "label_mapping": {
            "normal": 0,
            "nmap_10_T5": 1,
            "qos_mid_ddos": 1,
            "brute_force_malformed": 1,
            "qos_mid": 1,
            "nmap_sub": 1,
            "nmap_mqtt": 1,
            "nmap_banner": 1,
            "mqtt_cat": 1,
            "scp_inst": 1,
        },
    },
    "train_slash_char": {
        "label_mapping": {
            "normal": 0,
            "nmap_10_T4": 1,
            "slash_char": 1,
            "nmap_10_T5": 1,
            "brute_force_timing": 1,
            "nmap_mqtt": 1,
            "nmap_banner": 1,
            "nmap_sub": 1,
            "mqtt_cat": 1,
            "sftp_inst": 1,
        },
    },
    "train_sub_exf": {
        "label_mapping": {
            "normal": 0,
            "nmap_10_T5": 1,
            "brute_force_timing": 1,
            "nmap_banner": 1,
            "nmap_sub": 1,
            "scp_exf": 1,
            "nmap_mqtt": 1,
            "mqtt_cat": 1,
        },
    },
}
SCHEMAS = {}
for scenario in SCENARIOS:
    settings = {**COMMON_SCHEMA, **SCHEMA_OVERRIDES.get(scenario, {})}
    unresolved = [key for key, value in settings.items() if value is None]
    if unresolved or not settings.get("label_mapping"):
        raise ValueError(f"Resolve schema fields for {scenario}: {unresolved}")
    schema = AuditSchema(**settings)
    if scenario in INSPECTIONS:
        schema.validate(INSPECTIONS[scenario]["columns"])
    SCHEMAS[scenario] = schema
print("All selected scenario mappings are explicit.")


All selected scenario mappings are explicit.


## 6. Run the complete scenario audit

This cell scans each selected CSV completely. It downloads or reuses one raw file locally, writes compressed audit Parquet, and computes disk-backed statistics for 1, 5, 10, and 30 seconds, including half-window origin shifts. Epoch-aligned boundaries are diagnostic candidates, not a frozen window choice.

Repeated packets remain distinct rows. Duplicate-row and duplicate-column detection uses SHA-256 fingerprints. Numeric parse failures may indicate legitimate categorical fields; inspect them before declaring malformed data. Window distributions describe occupied windows; empty windows between the first and last packet are counted separately.

Report files are copied to Drive and checksum-verified. Only then is that scenario's isolated local workspace deleted. The authors' originals are never modified. The preview download is reused and removed only after its results are safely persisted. A failure retains local work and records its location on Drive; a data-integrity blocker stops before the next scenario. SQL spill space can exceed raw size, so monitor local storage.

An audit with no automatic blockers still requires review of endpoints, sequence semantics, timestamps, duplicates, feature leakage, and window feasibility.


In [ ]:
RESULTS = run_gate0(
    manifest_path=MANIFEST_PATH,
    mode=MODE,
    sources=SOURCES,
    schemas=SCHEMAS,
    local_root=LOCAL_ROOT,
    drive_run_dir=DRIVE_RUN_DIR,
    smoke_review_path=Path(SMOKE_REVIEW_PATH) if SMOKE_REVIEW_PATH else None,
    chunksize=CHUNK_SIZE,
    memory_limit=DUCKDB_MEMORY_LIMIT,
    threads=DUCKDB_THREADS,
    keep_audit_parquet=KEEP_AUDIT_PARQUET,
    source_cache_root=SOURCE_CACHE_ROOT,
)
print(json.dumps(RESULTS, indent=2))


Starting train_empty_conn. Local workspace: /content/capture_gate0_work/train_empty_conn_zk6tz9ox
Reusing verified local source: normal_empty_conn_train.csv
train_empty_conn: converted 25,000 packets
train_empty_conn: converted 50,000 packets
train_empty_conn: converted 75,000 packets
train_empty_conn: converted 100,000 packets
train_empty_conn: converted 125,000 packets
train_empty_conn: converted 150,000 packets
train_empty_conn: converted 175,000 packets
train_empty_conn: converted 200,000 packets
train_empty_conn: converted 225,000 packets
train_empty_conn: converted 250,000 packets
train_empty_conn: converted 275,000 packets
train_empty_conn: converted 300,000 packets
train_empty_conn: converted 325,000 packets
train_empty_conn: converted 350,000 packets
train_empty_conn: converted 375,000 packets
train_empty_conn: converted 400,000 packets
train_empty_conn: converted 425,000 packets
train_empty_conn: converted 450,000 packets
train_empty_conn: converted 475,000 packets
train_empt

Downloading...
From (original): https://drive.google.com/uc?id=1syQC83AZTi-J6VHW2g1p__vqKQorXq6l
From (redirected): https://drive.google.com/uc?id=1syQC83AZTi-J6VHW2g1p__vqKQorXq6l&confirm=t&uuid=17bd535e-371b-47db-a390-635975e48af3
To: /content/capture_gate0_work/20260917T170658_227477Z_full_dev/sources/train_qos_mid/normal_qos_mid_train.csv.part
100%|██████████| 1.04G/1.04G [00:35<00:00, 29.0MB/s]


train_qos_mid: converted 25,000 packets
train_qos_mid: converted 50,000 packets
train_qos_mid: converted 75,000 packets
train_qos_mid: converted 100,000 packets
train_qos_mid: converted 125,000 packets
train_qos_mid: converted 150,000 packets
train_qos_mid: converted 175,000 packets
train_qos_mid: converted 200,000 packets
train_qos_mid: converted 225,000 packets
train_qos_mid: converted 250,000 packets
train_qos_mid: converted 275,000 packets
train_qos_mid: converted 300,000 packets
train_qos_mid: converted 325,000 packets
train_qos_mid: converted 350,000 packets
train_qos_mid: converted 375,000 packets
train_qos_mid: converted 400,000 packets
train_qos_mid: converted 425,000 packets
train_qos_mid: converted 450,000 packets
train_qos_mid: converted 475,000 packets
train_qos_mid: converted 500,000 packets
train_qos_mid: converted 525,000 packets
train_qos_mid: converted 550,000 packets
train_qos_mid: converted 575,000 packets
train_qos_mid: converted 600,000 packets
train_qos_mid: conv

Downloading...
From (original): https://drive.google.com/uc?id=1N3IcqQz6tQe-iuvoqWWwVarPCJDK_cpe
From (redirected): https://drive.google.com/uc?id=1N3IcqQz6tQe-iuvoqWWwVarPCJDK_cpe&confirm=t&uuid=1870c2a3-9f6f-4242-ada6-9464bc163fdd
To: /content/capture_gate0_work/20260917T170658_227477Z_full_dev/sources/train_dollar_char/normal_dollar_char_train.csv.part
100%|██████████| 1.97G/1.97G [00:41<00:00, 47.3MB/s]


train_dollar_char: converted 25,000 packets
train_dollar_char: converted 50,000 packets
train_dollar_char: converted 75,000 packets
train_dollar_char: converted 100,000 packets
train_dollar_char: converted 125,000 packets
train_dollar_char: converted 150,000 packets
train_dollar_char: converted 175,000 packets
train_dollar_char: converted 200,000 packets
train_dollar_char: converted 225,000 packets
train_dollar_char: converted 250,000 packets
train_dollar_char: converted 275,000 packets
train_dollar_char: converted 300,000 packets
train_dollar_char: converted 325,000 packets
train_dollar_char: converted 350,000 packets
train_dollar_char: converted 375,000 packets
train_dollar_char: converted 400,000 packets
train_dollar_char: converted 425,000 packets
train_dollar_char: converted 450,000 packets
train_dollar_char: converted 475,000 packets
train_dollar_char: converted 500,000 packets
train_dollar_char: converted 525,000 packets
train_dollar_char: converted 550,000 packets
train_dollar_

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved and verified train_slash_char on Drive. Removed its temporary local copy.
Starting train_sub_exf. Local workspace: /content/capture_gate0_work/train_sub_exf_h449vl_u


Downloading...
From (original): https://drive.google.com/uc?id=1YJzcazSvs987g1xPWal8gxBpqeYfcSp4
From (redirected): https://drive.google.com/uc?id=1YJzcazSvs987g1xPWal8gxBpqeYfcSp4&confirm=t&uuid=1ccedb33-a498-4c0e-8a12-27fb38a1a30a
To: /content/capture_gate0_work/20260917T170658_227477Z_full_dev/sources/train_sub_exf/normal_sub_exf_train.csv.part
100%|██████████| 1.66G/1.66G [00:46<00:00, 35.7MB/s]


train_sub_exf: converted 25,000 packets
train_sub_exf: converted 50,000 packets
train_sub_exf: converted 75,000 packets
train_sub_exf: converted 100,000 packets
train_sub_exf: converted 125,000 packets
train_sub_exf: converted 150,000 packets
train_sub_exf: converted 175,000 packets
train_sub_exf: converted 200,000 packets
train_sub_exf: converted 225,000 packets
train_sub_exf: converted 250,000 packets
train_sub_exf: converted 275,000 packets
train_sub_exf: converted 300,000 packets
train_sub_exf: converted 325,000 packets
train_sub_exf: converted 350,000 packets
train_sub_exf: converted 375,000 packets
train_sub_exf: converted 400,000 packets
train_sub_exf: converted 425,000 packets
train_sub_exf: converted 450,000 packets
train_sub_exf: converted 475,000 packets
train_sub_exf: converted 500,000 packets
train_sub_exf: converted 525,000 packets
train_sub_exf: converted 550,000 packets
train_sub_exf: converted 575,000 packets
train_sub_exf: converted 600,000 packets
train_sub_exf: conv

## 7. Review the saved reports

The run directory contains the manifest snapshot, resolved runtime configuration, Git provenance, run status, and per-scenario artifacts:

- `audit_report.json`: schema, packet counts, raw labels, column diagnostics, blockers, and window summaries.
- `packets.audit.parquet`: all rows and raw fields, plus canonical diagnostic metadata (optional durable retention).
- `attack_iterations.parquet`: counts and durations grouped by attack step, phase, and sequence.
- `windows_*s_offset_*s.parquet`: occupied-window packet, node, and directed-pair counts.
- `artifact_checksums.json`: hashes of the persisted files.

A zero-duration attack iteration is possible when all its packets have the same timestamp. Duplicate benign data across scenarios sharing the same benign source is not removed; the manifest's background-separated folds address that evaluation risk.


In [ ]:
REPORTS = {}
for scenario, result in RESULTS.items():
    report = json.loads(Path(result["report"]).read_text())
    REPORTS[scenario] = report
    print(f"\n{scenario}: {report['status']}")
    print(f"Blockers: {report['blockers']}")
    display(pd.DataFrame([report["counts"]]))
    display(pd.DataFrame(report["raw_labels"]))
    window_summary = pd.DataFrame(report["windows"])
    window_summary["mean_packets_per_second_occupied"] = (
        window_summary["mean_packets_occupied"] / window_summary["width_seconds"]
    )
    display(window_summary)
    display(pd.DataFrame(report["column_profiles"]).T)
    print("Duplicate column groups:", report["duplicate_column_groups_sha256"])
    iterations = pd.read_parquet(Path(result["report"]).parent / "attack_iterations.parquet")
    display(iterations.head(50))



train_empty_conn: review_required
Blockers: []


,packets,normal_packets,attack_packets,unmapped_labels,invalid_timestamps,missing_endpoints,incomplete_attack_annotations,first_timestamp_seconds,last_timestamp_seconds,duplicate_raw_rows_sha256,unique_endpoints,unique_directed_pairs
0,1175779,746806,428973,0,0,0,0,0.019792,46799.813163,0,160,623


,raw_label,packets
0,normal,746806
1,nmap_10_T4,255667
2,brute_force_timing,107515
3,empty_conn_ddos,28269
4,nmap_mqtt,14168
5,nmap_banner,12649
6,empty_conn,6650
7,nmap_sub,2277
8,mqtt_cat,1171
9,sftp_inst,607


,width_seconds,origin_offset_seconds,artifact,occupied_windows,empty_windows_between_first_and_last,mean_packets_occupied,max_packets,packet_quantiles_occupied,max_nodes,node_quantiles_occupied,max_directed_pairs,pair_quantiles_occupied,mean_packets_per_second_occupied
0,1,0.0,windows_1s_offset_0s.parquet,44480,2320,26.433880,15524,"[16.0, 37.0, 145.41999999999825]",56,"[4.0, 8.0, 10.0]",96,"[5.0, 11.0, 14.0]",26.433880
1,1,0.5,windows_1s_offset_0.5s.parquet,44435,2366,26.460650,15616,"[16.0, 37.0, 146.0]",67,"[4.0, 8.0, 11.0]",92,"[5.0, 11.0, 14.659999999996217]",26.460650
2,5,0.0,windows_5s_offset_0s.parquet,9360,0,125.617415,42145,"[79.0, 134.04999999999927, 616.4099999999999]",86,"[13.0, 20.0, 33.0]",108,"[20.0, 31.0, 40.409999999999854]",25.123483
3,5,2.5,windows_5s_offset_2.5s.parquet,9361,0,125.603995,42148,"[79.0, 132.0, 616.1999999999989]",80,"[13.0, 20.0, 33.0]",106,"[20.0, 31.0, 40.0]",25.120799
4,10,0.0,windows_10s_offset_0s.parquet,4680,0,251.234829,42245,"[159.0, 261.10000000000036, 920.0]",88,"[15.0, 22.0, 37.0]",111,"[24.0, 37.0, 62.0]",25.123483
5,10,5.0,windows_10s_offset_5s.parquet,4681,0,251.181158,42231,"[161.0, 267.0, 874.1999999999998]",86,"[15.0, 22.0, 37.0]",115,"[24.0, 36.0, 60.19999999999982]",25.118116
6,30,0.0,windows_30s_offset_0s.parquet,1560,0,753.704487,42592,"[490.5, 1032.1, 1678.4400000000069]",95,"[19.0, 37.0, 45.0]",162,"[30.0, 52.0, 74.23000000000025]",25.123483
7,30,15.0,windows_30s_offset_15s.parquet,1561,0,753.221653,43255,"[492.0, 1027.0, 1633.0000000000018]",93,"[19.0, 38.0, 45.40000000000009]",132,"[31.0, 53.0, 77.0]",25.107388


,missing,numeric_values,constant_nonmissing,numeric_parse_failures_nonmissing,interpretation
layers_frame_frame.section_number,0,1175779,True,0,Parse failures may be valid categorical values...
timestamp,0,0,False,1175779,Parse failures may be valid categorical values...
layers_frame_frame.time_epoch,0,1175779,False,0,Parse failures may be valid categorical values...
layers_frame_frame.number,0,1175779,False,0,Parse failures may be valid categorical values...
layers_frame_frame.len,0,1175779,False,0,Parse failures may be valid categorical values...
...,...,...,...,...,...
phase_idx,746806,428973,False,0,Parse failures may be valid categorical values...
phase_name,746806,0,False,428973,Parse failures may be valid categorical values...
phase_number,746806,428973,False,0,Parse failures may be valid categorical values...
step_number,746806,428973,False,0,Parse failures may be valid categorical values...


...

,raw_label,packets
0,normal,1705003
1,nmap_10_T5,255580
2,brute_force_timing,107640
3,nmap_banner,79052
4,nmap_sub,67366
5,scp_exf,5764
6,nmap_mqtt,2852
7,mqtt_cat,2550


,width_seconds,origin_offset_seconds,artifact,occupied_windows,empty_windows_between_first_and_last,mean_packets_occupied,max_packets,packet_quantiles_occupied,max_nodes,node_quantiles_occupied,max_directed_pairs,pair_quantiles_occupied,mean_packets_per_second_occupied
0,1,0.0,windows_1s_offset_0s.parquet,39538,7259,56.295387,15427,"[30.0, 165.15000000000146, 458.0]",57,"[5.0, 9.0, 25.0]",94,"[8.0, 14.0, 30.0]",56.295387
1,1,0.5,windows_1s_offset_0.5s.parquet,39547,7251,56.282575,16135,"[30.0, 167.0, 450.0]",68,"[5.0, 9.0, 26.0]",94,"[8.0, 14.0, 30.0]",56.282575
2,5,0.0,windows_5s_offset_0s.parquet,9116,244,244.164875,42250,"[165.0, 620.0, 1103.0000000000073]",85,"[13.0, 32.0, 37.0]",112,"[21.0, 41.0, 66.0]",48.832975
3,5,2.5,windows_5s_offset_2.5s.parquet,9118,242,244.111318,42326,"[166.0, 605.2999999999993, 1045.3199999999997]",79,"[13.0, 31.149999999999636, 37.0]",105,"[21.0, 40.149999999999636, 66.0]",48.822264
4,10,0.0,windows_10s_offset_0s.parquet,4680,0,475.599786,42685,"[357.0, 1028.1500000000005, 1749.21]",96,"[15.0, 37.0, 40.0]",123,"[25.0, 58.05000000000018, 72.0]",47.559979
5,10,5.0,windows_10s_offset_5s.parquet,4681,0,475.498184,42804,"[358.0, 1025.0, 1641.3999999999996]",85,"[15.0, 37.0, 40.0]",112,"[25.0, 57.0, 74.0]",47.549818
6,30,0.0,windows_30s_offset_0s.parquet,1560,0,1426.799359,43793,"[1276.5, 2524.45, 3936.1300000000074]",97,"[20.0, 43.0, 51.0]",170,"[33.0, 78.0, 84.0]",47.559979
7,30,15.0,windows_30s_offset_15s.parquet,1561,0,1425.885330,43900,"[1264.0, 2498.0, 3988.4000000000087]",97,"[20.0, 42.0, 51.0]",131,"[34.0, 78.0, 84.40000000000009]",47.529511


,missing,numeric_values,constant_nonmissing,numeric_parse_failures_nonmissing,interpretation
layers_frame_frame.section_number,0,2225807,True,0,Parse failures may be valid categorical values...
timestamp,0,0,False,2225807,Parse failures may be valid categorical values...
layers_frame_frame.time_epoch,0,2225807,False,0,Parse failures may be valid categorical values...
layers_frame_frame.number,0,2225807,False,0,Parse failures may be valid categorical values...
layers_frame_frame.len,0,2225807,False,0,Parse failures may be valid categorical values...
...,...,...,...,...,...
phase_idx,1705003,520804,False,0,Parse failures may be valid categorical values...
phase_name,1705003,0,False,520804,Parse failures may be valid categorical values...
phase_number,1705003,520804,False,0,Parse failures may be valid categorical values...
step_number,1705003,520804,False,0,Parse failures may be valid categorical values...


Duplicate column groups: [['layers_frame_frame.len', 'layers_frame_frame.cap_len'], ['layers_eth_eth.dst', 'layers_eth_eth.dst_tree_eth.addr'], ['layers_eth_eth.dst_tree_eth.dst_resolved', 'layers_eth_eth.dst_tree_eth.addr_resolved'], ['layers_eth_eth.dst_tree_eth.dst.oui', 'layers_eth_eth.dst_tree_eth.addr.oui'], ['layers_eth_eth.src', 'layers_eth_eth.src_tree_eth.src_resolved'], ['layers_eth_eth.src_tree_eth.src.oui', 'layers_eth_eth.src_tree_eth.addr.oui'], ['layers_ipv6_ipv6.addr', 'layers_ipv6_ipv6.host'], ['layers_ip_ip.addr', 'layers_ip_ip.host'], ['layers_tcp_tcp.dstport', 'layers_tcp_tcp.port'], ['layers_ssh_SSH Version 2_Key Exchange_Algorithms_ssh.encryption_algorithms_client_to_server', 'layers_ssh_SSH Version 2_Key Exchange_Algorithms_ssh.encryption_algorithms_server_to_client'], ['layers_ssh_SSH Version 2_Key Exchange_Algorithms_ssh.mac_algorithms_client_to_server', 'layers_ssh_SSH Version 2_Key Exchange_Algorithms_ssh.mac_algorithms_server_to_client'], ['layers_ssh_SSH V

,attack_step,phase,sequence_id,packets,start_seconds,end_seconds,duration_seconds
0,nmap_10_T5,RECONNAISSANCE,2.15,42666,3171.125405,3226.938418,55.813013
1,nmap_10_T5,RECONNAISSANCE,4.15,42579,3290.338422,3346.309933,55.971511
2,nmap_10_T5,RECONNAISSANCE,6.15,42596,3405.306401,3461.190726,55.884325
3,nmap_10_T5,RECONNAISSANCE,8.15,42583,3497.126019,3553.553107,56.427088
4,nmap_10_T5,RECONNAISSANCE,10.15,42579,3618.369427,3674.331933,55.962506
5,nmap_10_T5,RECONNAISSANCE,12.15,42577,3704.979033,3760.717752,55.738719
6,brute_force_timing,BRUTE_FORCE,14.15,107640,3807.653315,4868.518765,1060.865450
7,mqtt_cat,DISCOVERY,44.15,91,6893.618145,6898.840303,5.222158
8,scp_exf,EXPLOIT,46.15,49,6956.763423,6956.947811,0.184388
9,nmap_sub,DISCOVERY,52.15,726,7258.745333,7313.121387,54.376054


In [ ]:
scenario_summary = []
window_summary = []
label_summary = []
column_sets = {}

for scenario, report in REPORTS.items():
    counts = report["counts"]
    column_sets[scenario] = set(report["column_profiles"])

    scenario_summary.append({
        "scenario": scenario,
        "status": report["status"],
        "blockers": ", ".join(report["blockers"]),
        "packets": counts["packets"],
        "normal_packets": counts["normal_packets"],
        "attack_packets": counts["attack_packets"],
        "attack_fraction": (
            counts["attack_packets"] / counts["packets"]
        ),
        "unique_endpoints": counts["unique_endpoints"],
        "unique_directed_pairs": counts["unique_directed_pairs"],
        "invalid_timestamps": counts["invalid_timestamps"],
        "timestamp_inversions": report["timestamp_order_inversions"],
        "unmapped_labels": counts["unmapped_labels"],
        "missing_endpoints": counts["missing_endpoints"],
        "incomplete_attack_annotations": (
            counts["incomplete_attack_annotations"]
        ),
        "duplicate_raw_rows": counts["duplicate_raw_rows_sha256"],
        "source_gib": report["source_size_bytes"] / 1024**3,
        "audit_parquet_gib": (
            report["audit_parquet_size_bytes"] / 1024**3
        ),
        "columns": len(report["column_profiles"]),
        "duplicate_column_groups": len(
            report["duplicate_column_groups_sha256"]
        ),
    })

    for item in report["raw_labels"]:
        label_summary.append({
            "scenario": scenario,
            **item,
        })

    for item in report["windows"]:
        window_summary.append({
            "scenario": scenario,
            "width_seconds": item["width_seconds"],
            "origin_offset_seconds": item["origin_offset_seconds"],
            "occupied_windows": item["occupied_windows"],
            "empty_windows": (
                item["empty_windows_between_first_and_last"]
            ),
            "median_packets": item["packet_quantiles_occupied"][0],
            "p95_packets": item["packet_quantiles_occupied"][1],
            "p99_packets": item["packet_quantiles_occupied"][2],
            "max_packets": item["max_packets"],
            "p99_nodes": item["node_quantiles_occupied"][2],
            "max_nodes": item["max_nodes"],
            "p99_directed_pairs": item[
                "pair_quantiles_occupied"
            ][2],
            "max_directed_pairs": item["max_directed_pairs"],
        })

scenario_summary = pd.DataFrame(scenario_summary)
label_summary = pd.DataFrame(label_summary)
window_summary = pd.DataFrame(window_summary)

all_columns = set.union(*column_sets.values())
common_columns = set.intersection(*column_sets.values())

schema_differences = {
    scenario: {
        "missing_from_union": sorted(all_columns - columns),
        "scenario_specific": sorted(
            columns - set.union(
                *[
                    other_columns
                    for other_scenario, other_columns
                    in column_sets.items()
                    if other_scenario != scenario
                ]
            )
        ),
    }
    for scenario, columns in column_sets.items()
}

iteration_summaries = []

for scenario, result in RESULTS.items():
    artifact_directory = Path(result["report"]).parent
    iterations = pd.read_parquet(
        artifact_directory / "attack_iterations.parquet"
    )

    sequence_cardinality = iterations.groupby("sequence_id").agg(
        attack_steps=("attack_step", "nunique"),
        phases=("phase", "nunique"),
    )
    conflicts = sequence_cardinality[
        (sequence_cardinality["attack_steps"] > 1)
        | (sequence_cardinality["phases"] > 1)
    ]

    by_step = (
        iterations.groupby(["attack_step", "phase"], as_index=False)
        .agg(
            iterations=("sequence_id", "nunique"),
            packets=("packets", "sum"),
            min_duration_seconds=("duration_seconds", "min"),
            median_duration_seconds=("duration_seconds", "median"),
            max_duration_seconds=("duration_seconds", "max"),
        )
    )
    by_step.insert(0, "scenario", scenario)
    by_step["sequence_conflicts"] = len(conflicts)
    iteration_summaries.append(by_step)

attack_step_summary = pd.concat(
    iteration_summaries,
    ignore_index=True,
)

print("Scenario summary")
display(scenario_summary)

print("Raw label summary")
display(label_summary)

print("Window summary")
display(window_summary)

print(f"Union columns: {len(all_columns)}")
print(f"Common columns: {len(common_columns)}")
print("Schema differences")
print(json.dumps(schema_differences, indent=2))

print("Attack-step summary")
display(attack_step_summary)



Scenario summary


,scenario,status,blockers,packets,normal_packets,attack_packets,attack_fraction,unique_endpoints,unique_directed_pairs,invalid_timestamps,timestamp_inversions,unmapped_labels,missing_endpoints,incomplete_attack_annotations,duplicate_raw_rows,source_gib,audit_parquet_gib,columns,duplicate_column_groups
0,train_empty_conn,review_required,,1175779,746806,428973,0.364842,160,623,0,0,0,0,0,0,0.960427,0.248701,108,12
1,train_qos_mid,review_required,,1499717,746806,752911,0.502035,163,481,0,0,0,0,0,0,0.969574,0.198402,107,12
2,train_dollar_char,review_required,,2882555,1705003,1177552,0.408510,160,633,0,0,0,0,0,0,1.833687,0.375520,107,12
3,train_slash_char,review_required,,6425516,1705003,4720513,0.734651,164,435,0,0,0,0,0,0,4.745214,0.698431,108,12
4,train_sub_exf,review_required,,2225807,1705003,520804,0.233984,121,497,0,0,0,0,0,0,1.545524,0.356730,107,12


Raw label summary


,scenario,raw_label,packets
0,train_empty_conn,normal,746806
1,train_empty_conn,nmap_10_T4,255667
2,train_empty_conn,brute_force_timing,107515
3,train_empty_conn,empty_conn_ddos,28269
4,train_empty_conn,nmap_mqtt,14168
5,train_empty_conn,nmap_banner,12649
6,train_empty_conn,empty_conn,6650
7,train_empty_conn,nmap_sub,2277
8,train_empty_conn,mqtt_cat,1171
9,train_empty_conn,sftp_inst,607


Window summary


,scenario,width_seconds,origin_offset_seconds,occupied_windows,empty_windows,median_packets,p95_packets,p99_packets,max_packets,p99_nodes,max_nodes,p99_directed_pairs,max_directed_pairs
0,train_empty_conn,1,0.0,44480,2320,16.0,37.00,145.42,15524,10.0,56,14.00,96
1,train_empty_conn,1,0.5,44435,2366,16.0,37.00,146.00,15616,11.0,67,14.66,92
2,train_empty_conn,5,0.0,9360,0,79.0,134.05,616.41,42145,33.0,86,40.41,108
3,train_empty_conn,5,2.5,9361,0,79.0,132.00,616.20,42148,33.0,80,40.00,106
4,train_empty_conn,10,0.0,4680,0,159.0,261.10,920.00,42245,37.0,88,62.00,111
5,train_empty_conn,10,5.0,4681,0,161.0,267.00,874.20,42231,37.0,86,60.20,115
6,train_empty_conn,30,0.0,1560,0,490.5,1032.10,1678.44,42592,45.0,95,74.23,162
7,train_empty_conn,30,15.0,1561,0,492.0,1027.00,1633.00,43255,45.4,93,77.00,132
8,train_qos_mid,1,0.0,45154,1646,21.0,45.00,75.00,14815,11.0,56,16.00,96
9,train_qos_mid,1,0.5,45107,1694,21.0,45.00,75.00,14642,11.0,67,16.00,92


Union columns: 108
Common columns: 107
Schema differences
{
  "train_empty_conn": {
    "missing_from_union": [],
    "scenario_specific": []
  },
  "train_qos_mid": {
    "missing_from_union": [
      "layers_tcp_tcp.options_tree_tcp.options.sack"
    ],
    "scenario_specific": []
  },
  "train_dollar_char": {
    "missing_from_union": [
      "layers_tcp_tcp.options_tree_tcp.options.sack"
    ],
    "scenario_specific": []
  },
  "train_slash_char": {
    "missing_from_union": [],
    "scenario_specific": []
  },
  "train_sub_exf": {
    "missing_from_union": [
      "layers_tcp_tcp.options_tree_tcp.options.sack"
    ],
    "scenario_specific": []
  }
}
Attack-step summary


,scenario,attack_step,phase,iterations,packets,min_duration_seconds,median_duration_seconds,max_duration_seconds,sequence_conflicts
0,train_empty_conn,brute_force_timing,BRUTE_FORCE,1,107515,1048.432440,1048.432440,1048.432440,0
1,train_empty_conn,empty_conn,EXPLOIT,37,6650,76.621535,95.356958,114.146406,0
2,train_empty_conn,empty_conn_ddos,EXPLOIT,7,28269,2394.303065,2450.593708,2464.030255,0
3,train_empty_conn,mqtt_cat,DISCOVERY,12,1171,5.206165,5.223312,5.789674,0
4,train_empty_conn,nmap_10_T4,RECONNAISSANCE,6,255667,56.092985,56.684457,56.845175,0
5,train_empty_conn,nmap_banner,DISCOVERY,14,12649,64.334165,64.359124,65.149329,0
6,train_empty_conn,nmap_mqtt,DISCOVERY,15,14168,242.378472,242.401024,242.453249,0
7,train_empty_conn,nmap_sub,DISCOVERY,3,2277,54.377392,54.386559,54.397186,0
8,train_empty_conn,sftp_inst,INSTALLATION,12,607,0.153654,0.164405,0.168971,0
9,train_qos_mid,brute_force_malformed,BRUTE_FORCE,2,30860,860.257237,876.992471,893.727705,0


## 8. Record the smoke review before FULL_DEV

Do this only after both smoke scenarios finish and their scientific checks have been reviewed. Record the rationale and any unresolved choices. This review permits the full development audit; it does not freeze features, pass a modeling gate, or authorize final-test access.

The review binds the exact manifest and report hashes. Changing the manifest invalidates it. Keep `APPROVE_SMOKE=False` until review is complete. To run `FULL_DEV`, return to section 3, change the mode, use the five sources already configured in the manifest, and set `SMOKE_REVIEW_PATH` to the printed path.


In [ ]:
APPROVE_SMOKE = False
REVIEW_NOTES = ""


if APPROVE_SMOKE:
    if MODE != "SMOKE":
        raise ValueError("Create smoke reviews only from SMOKE runs.")
    expected = selected_scenarios(MANIFEST, "SMOKE")
    if set(REPORTS) != set(expected) or any(report["blockers"] for report in REPORTS.values()):
        raise ValueError("Both smoke reports must exist and have no automatic blockers.")
    if not REVIEW_NOTES.strip():
        raise ValueError("Record the review rationale before approval.")
    review = {
        "approved": True,
        "manifest_sha256": sha256_file(MANIFEST_PATH),
        "review_notes": REVIEW_NOTES,
        "reports": {
            scenario: {
                "path": RESULTS[scenario]["report"],
                "sha256": sha256_file(Path(RESULTS[scenario]["report"])),
            }
            for scenario in expected
        },
    }
    review_path = DRIVE_RUN_DIR / "smoke_review.json"
    if review_path.exists():
        raise FileExistsError("A smoke review already exists; preserve the original decision.")
    write_json(review_path, review)
    validate_smoke_review(review_path, sha256_file(MANIFEST_PATH), MANIFEST)
    print(f"Smoke review saved: {review_path}")
else:
    print("Smoke review remains pending. No approval was recorded.")


## 9. Audit the remaining Gate-0 decisions

Run this section only for the completed `FULL_DEV` artifact collection. It reads the persisted audit Parquet files directly from Drive and does not download source CSVs.

The analysis:

- compares normal-packet streams among scenarios that declare the same benign source;
- counts broadcast, multicast, self-loop, IPv4, IPv6, and non-IP packets;
- builds a feature inventory with diagnostic inclusion and exclusion suggestions.

Benign equality uses SHA-256 over an ordered stream of DuckDB 64-bit row hashes computed from every raw column shared within a benign-source group. This is a high-confidence data-level equality diagnostic rather than a byte-for-byte comparison of the original CSV files.

Feature actions are proposals for manual review. Running this section does not freeze the node key, feature schema, sampling policy, or window duration.


In [4]:
decision_test_environment = dict(os.environ)
decision_test_environment["PYTHONPATH"] = str(PROJECT_ROOT / "code/python")
subprocess.run(
    [sys.executable, "-m", "unittest", "discover",
     "-s", str(PROJECT_ROOT / "code/python/tests"),
     "-p", "test_capture_gate0_review.py", "-v"],
    env=decision_test_environment,
    cwd=PROJECT_ROOT,
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', 'discover', '-s', '/content/temporalgnn-nids/code/python/tests', '-p', 'test_capture_gate0_review.py', '-v'], returncode=0)

In [5]:
from utils import capture_gate0_review

FULL_DEV_RUN_DIR = (
    DRIVE_ROOT / "runs" / "20260917T170658_227477Z_full_dev"
)
DECISION_AUDIT_PATH = FULL_DEV_RUN_DIR / "gate0_decision_audit.json"
DECISION_MODULE_SHA256 = sha256_file(Path(capture_gate0_review.__file__))

if DECISION_AUDIT_PATH.exists():
    DECISION_AUDIT = json.loads(DECISION_AUDIT_PATH.read_text())
    if DECISION_AUDIT["manifest_sha256"] != sha256_file(MANIFEST_PATH):
        raise ValueError("The existing decision audit belongs to a different manifest.")
    if DECISION_AUDIT.get("module_sha256") != DECISION_MODULE_SHA256:
        raise ValueError("The existing decision audit was produced by a different module.")
    print(f"Reusing existing decision audit: {DECISION_AUDIT_PATH}")
else:
    DECISION_AUDIT = capture_gate0_review.run_gate0_decision_audit(
        manifest_path=MANIFEST_PATH,
        run_dir=FULL_DEV_RUN_DIR,
        memory_limit=DUCKDB_MEMORY_LIMIT,
        threads=DUCKDB_THREADS,
    )
    print(f"Decision audit saved: {DECISION_AUDIT_PATH}")

print(f"Decision-audit status: {DECISION_AUDIT['status']}")
print(f"Blockers: {DECISION_AUDIT['blockers']}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Decision audit saved: /content/drive/MyDrive/capture_gate0/runs/20260917T170658_227477Z_full_dev/gate0_decision_audit.json
Decision-audit status: review_required
Blockers: []


In [6]:
benign_rows = []
for comparison in DECISION_AUDIT["benign_background_comparison"]:
    for scenario, signature in comparison["signatures"].items():
        benign_rows.append({
            "benign_source": comparison["benign_source"],
            "scenario": scenario,
            "all_group_scenarios_identical": comparison[
                "all_benign_packets_identical"
            ],
            "benign_packets": signature["benign_packets"],
            "common_raw_columns": signature["common_raw_columns"],
            "signature": signature[
                "sha256_of_ordered_duckdb_row_hashes"
            ],
        })

topology_rows = []
for item in DECISION_AUDIT["topology"]:
    topology_rows.append({
        key: value
        for key, value in item.items()
        if key != "top_group_destinations"
    })

feature_inventory = pd.DataFrame(
    DECISION_AUDIT["feature_inventory"]
)
feature_review = feature_inventory[
    feature_inventory["suggested_action"] != "candidate_numeric"
][[
    "column",
    "suggested_role",
    "suggested_action",
    "missing_scenarios",
    "missing_values",
    "numeric_when_present",
    "constant_nonmissing_in_each_present_scenario",
]]

print("Benign-background comparison")
display(pd.DataFrame(benign_rows))

print("Topology summary")
display(pd.DataFrame(topology_rows))

print("Top broadcast and multicast destinations")
for item in DECISION_AUDIT["topology"]:
    print(item["scenario"])
    display(pd.DataFrame(item["top_group_destinations"]))

print("Feature action counts")
display(pd.DataFrame(
    sorted(DECISION_AUDIT["feature_action_counts"].items()),
    columns=["suggested_action", "columns"],
))

print("Features requiring exclusion or manual review")
display(feature_review)


Benign-background comparison


,benign_source,scenario,all_group_scenarios_identical,benign_packets,common_raw_columns,signature
0,normal_2_3_4,train_empty_conn,True,746806,107,28bfecad09ed1d4a075471a2edbe71c062fa355d4e6a5e...
1,normal_2_3_4,train_qos_mid,True,746806,107,28bfecad09ed1d4a075471a2edbe71c062fa355d4e6a5e...
2,normal_15_16_17,train_dollar_char,True,1705003,107,5a72511934141b602937a8cfec22b816e69f7e1a561bf2...
3,normal_15_16_17,train_slash_char,True,1705003,107,5a72511934141b602937a8cfec22b816e69f7e1a561bf2...
4,normal_15_16_17,train_sub_exf,True,1705003,107,5a72511934141b602937a8cfec22b816e69f7e1a561bf2...


Topology summary


,scenario,packets,invalid_mac_packets,self_loop_packets,broadcast_destination_packets,multicast_destination_packets,unicast_destination_packets,ipv4_packets,ipv6_packets,non_ip_packets,conflicting_ip_version_packets,arp_protocol_packets,unique_sources,unique_destinations
0,train_empty_conn,1175779,0,0,19805,27000,1128974,1099512,27000,49267,0,49267,131,73
1,train_qos_mid,1499717,0,0,20980,26240,1452497,1426456,26240,47021,0,47021,134,73
2,train_dollar_char,2882555,0,0,26334,23599,2832622,2801360,23599,57596,0,57596,131,73
3,train_slash_char,6425516,0,0,53481,22114,6349921,6320850,22114,82552,0,82552,135,73
4,train_sub_exf,2225807,0,0,94429,22280,2109098,2070845,22280,132682,0,132682,92,73


Top broadcast and multicast destinations
train_empty_conn


...

,destination,packets
0,ff:ff:ff:ff:ff:ff,94429
1,33:33:00:00:00:01,19860
2,33:33:00:00:00:fb,1313
3,33:33:00:00:00:02,574
4,33:33:00:00:00:16,418
5,33:33:00:00:00:0c,92
6,33:33:ff:09:0c:2c,1
7,33:33:ff:32:5e:9f,1
8,33:33:ff:35:65:f2,1
9,33:33:ff:40:02:8d,1


Feature action counts


,suggested_action,columns
0,candidate_categorical_or_mixed,15
1,candidate_numeric,39
2,exclude_absolute_time_feature,3
3,exclude_evaluation_metadata,6
4,exclude_exact_duplicate,2
5,exclude_primary_review_for_secondary_ablation,10
6,exclude_record_identifier,2
7,exclude_schema_inconsistent,1
8,fit_fold_variance_filter,15
9,topology_only_exclude_model_feature,15


Features requiring exclusion or manual review


,column,suggested_role,suggested_action,missing_scenarios,missing_values,numeric_when_present,constant_nonmissing_in_each_present_scenario
0,label,evaluation_metadata,exclude_evaluation_metadata,[],0,False,False
1,layers__ws.malformed__ws.malformed,candidate_packet_feature,fit_fold_variance_filter,[],14178732,False,True
2,layers_arp_arp.hw.type,candidate_packet_feature,fit_fold_variance_filter,[],13840256,True,True
3,layers_eth_eth.dst,topology_identity,topology_only_exclude_model_feature,[],0,False,False
4,layers_eth_eth.dst_tree_eth.addr,topology_identity,topology_only_exclude_model_feature,[],0,False,False
...,...,...,...,...,...,...,...
103,phase_name,evaluation_metadata,exclude_evaluation_metadata,[],6608621,False,False
104,phase_number,evaluation_metadata,exclude_evaluation_metadata,[],6608621,True,False
105,sequence_id,evaluation_metadata,exclude_evaluation_metadata,[],6608621,True,False
106,step_number,evaluation_metadata,exclude_evaluation_metadata,[],6608621,True,False
